# Worker A: Healthcare (All Conditions)

Runs the full Obermeyer healthcare grid in one session.
Healthcare uses analytic (KKT-based) gradients — no finite-difference overhead.

**Grid:** 7 methods × 2 alphas × 5 seeds = 70 runs

Checkpointed: re-run cells to resume from where you left off.

In [ ]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# Results directory
HC_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'healthcare')
os.makedirs(HC_RESULTS, exist_ok=True)

from experiments.colab_runner import *
import torch
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Results: {HC_RESULTS}')

## Configuration

Edit this cell to change any parameter. Re-run the cell, then re-run the experiment cell below.

In [ ]:
# ===== TASK PARAMETERS =====
TASK_OVERRIDES = {
    # 'n_sample': 0,             # 0 = all 48,784 patients
    # 'budget_rho': 0.35,        # budget = rho * sum(capped_costs)
    # 'test_fraction': 0.5,
    'val_fraction': 0.1,         # 10% validation split (for monitoring)
    # 'decision_mode': 'group',
    # 'fairness_type': 'mad',
}

# ===== TRAINING PARAMETERS =====
TRAIN_OVERRIDES = {
    # Optimizer
    'optimizer': 'adam',            # 'sgd' or 'adam'
    'lr': 0.001,                   # Learning rate
    'weight_decay': 1e-4,          # L2 regularization
    # Model architecture
    'init_mode': 'best_practice',  # Kaiming He initialization
    'dropout': 0.1,                # Dropout rate
    'hidden_dim': 64,              # MLP hidden width
    'n_layers': 2,                 # MLP depth
}

# ===== EXPERIMENT CONTROL =====
STEPS = 70
OVERWRITE = False

print('='*60)
print('HEALTHCARE EXPERIMENT CONFIGURATION')
print('='*60)
print(f'Steps per lambda: {STEPS}')
print(f'Overwrite: {OVERWRITE}')
print(f'Task overrides: {TASK_OVERRIDES}')
print(f'Train overrides: {TRAIN_OVERRIDES}')
print(f'Results dir: {HC_RESULTS}')
print('='*60)

## Run All Methods

In [ ]:
results = run_healthcare_slice(
    results_dir=HC_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    train_overrides=TRAIN_OVERRIDES,
    overwrite=OVERWRITE,
)

In [ ]:
show_progress(HC_RESULTS, 'Healthcare — COMPLETE')

if not results.empty:
    summary = results.groupby(['method_label', 'lambda', 'alpha_fair']).agg({
        'test_regret_normalized': ['mean', 'std'],
        'test_fairness': ['mean', 'std'],
        'test_pred_mse': ['mean', 'std'],
    }).round(4)
    print('\n--- Results Summary (mean +/- std over seeds) ---')
    print(summary.to_string())

Worker A complete. Run the Results notebook to analyze.